In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Nannuru BRAID-to-AlphaSimPy Tutorial

This notebook translates a BRAID breeding program abstraction into a runnable **AlphaSimPy** simulation.

The source BRAID program describes a **13-year varietal development pipeline** repeated across **3 cycles**:
- biparental crossing,
- sequential selfing from **F1 to F8**,
- preliminary, advanced, and elite yield testing,
- final variety release.

The original BRAID abstraction leaves several quantities unspecified, so this notebook uses explicit assumptions to create a coherent and executable AlphaSimPy workflow.

**Program**: Nannuru  
**Package**: AlphaSimPy


## BRAID Summary

**Key features extracted from the BRAID abstraction**:
- Horizon: **13 years**
- Repeated cycles: **3**
- Species model: **diploid line breeding**
- Trait architecture: **single additive target trait**
- Broad-sense information available in BRAID: trait heritability set to **0.3**
- Pipeline stages: **founders → F1 → F2 → F3 → F4 → F5 → F6 → F7 → F8 → preliminary trials → advanced trials → elite trials 1 → elite trials 2 → released variety**
- Selection in testing stages is **phenotypic truncation selection**


## Assumptions Used to Make the BRAID Program Executable

The BRAID file contains placeholders such as `size: variable`, `n_crosses: 0`, `chromosomes: 0`, and `n_qtl: 0`.
To create a runnable AlphaSimPy notebook, the following assumptions are used:

1. **Genome assumptions**
   - `n_chr = 1`
   - `n_qtl = 500`
   - `n_snp = 0`

2. **Founder assumptions**
   - `n_founders = 40` unrelated founder individuals generated by `runMacs`
   - founders are used as the external parent pool

3. **Crossing assumptions**
   - `n_crosses = 20` biparental crosses per cycle
   - `n_f1_per_cross = 1` F1 individual per cross

4. **Selfing assumptions**
   - each generation is advanced by selfing
   - `n_self_progeny = 10` progeny per parent per selfing generation
   - one generation of selfing is applied at each stage from F1→F2 through F7→F8

5. **Trial-stage assumptions**
   - preliminary, advanced, elite-1, and elite-2 stages are represented by phenotyping plus truncation selection
   - BRAID specifies selection intensity `0.2` for the first three trial transitions, so this notebook keeps the top **20%** at each of those stages
   - final release keeps the top **1** line
   - BRAID specifies `error_variance = 1.0`, which is used in `setPheno`

These assumptions are documented so the notebook remains transparent and easy to revise.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self,
    setPheno,
    selectInd,
    meanG,
    varG,
)

print("AlphaSimPy BRAID translation tutorial: Nannuru")
print("Libraries imported successfully.")


## Global Parameters

This section defines the executable simulation parameters corresponding to the BRAID abstraction and the assumptions listed above.

In [ ]:
# ---- Program structure ----
program_name = 'Nannuru'
n_cycles = 3
horizon_years = 13

# ---- Genome assumptions derived from BRAID placeholders ----
n_chr = 1
n_qtl = 500
n_snp = 0
trait_h2 = 0.3
var_e = 1.0

# ---- Founder and crossing assumptions ----
n_founders = 40
n_crosses = 20
n_f1_per_cross = 1
parents_per_cross = 2

# ---- Selfing pipeline assumptions ----
n_self_progeny = 10

# ---- Trial-stage selection assumptions ----
prelim_keep_frac = 0.20
advanced_keep_frac = 0.20
elite1_keep_frac = 0.20
n_release = 1

print('Simulation Parameters')
print(f'  Program: {program_name}')
print(f'  Cycles: {n_cycles}')
print(f'  Horizon (years): {horizon_years}')
print(f'  Chromosomes: {n_chr}')
print(f'  QTL per chromosome: {n_qtl}')
print(f'  Founder individuals: {n_founders}')
print(f'  Crosses per cycle: {n_crosses}')
print(f'  F1 per cross: {n_f1_per_cross}')
print(f'  Selfed progeny per parent per generation: {n_self_progeny}')
print(f'  Trial error variance: {var_e}')
print(f'  Trial selection fractions: {prelim_keep_frac}, {advanced_keep_frac}, {elite1_keep_frac}')
print(f'  Released lines per cycle: {n_release}')


## Create Founder Population and Simulation Parameters

We generate a founder population with `runMacs`, create a `SimParam` object, and define one additive trait.

In [ ]:
# Simulate founder haplotypes
founder_haplotypes = runMacs(nInd=n_founders, nChr=n_chr, segSites=n_qtl)

# Create simulation parameters
SP = SimParam(founder_haplotypes)
SP.addTraitA(nQtlPerChr=n_qtl)
SP.setVarE(h2=trait_h2)

# Create founder population
founders = newPop(founder_haplotypes, simParam=SP)

print('Founder population created.')
print(f'  Founder population size: {len(founders)}')
print(f'  Founder mean genetic value: {meanG(founders):.4f}')
print(f'  Founder genetic variance: {varG(founders):.4f}')


## Helper Functions

These helper functions keep the notebook readable and mirror the stage-based logic of the BRAID workflow.

In [ ]:
def keep_top_fraction(pop, frac, simParam):
    n_keep = max(1, int(np.ceil(len(pop) * frac)))
    return selectInd(pop, nInd=n_keep, use='pheno', simParam=simParam)

def phenotype_stage(pop, varE, simParam):
    setPheno(pop, varE=varE, simParam=simParam)
    return pop

def summarize_stage(cycle_id, year, stage_name, pop):
    return {
        'cycle': cycle_id,
        'year': year,
        'stage': stage_name,
        'n': len(pop),
        'meanG': float(meanG(pop)),
        'varG': float(varG(pop)),
    }


## Simulate the 13-Year Pipeline Across 3 Cycles

The BRAID workflow is implemented as a sequential pipeline:
1. make crosses,
2. self from F1 through F8,
3. phenotype and select in preliminary, advanced, elite-1, and elite-2 stages,
4. release the top line.

For simplicity, each cycle is simulated independently from the same founder pool.


In [ ]:
records = []
released_lines = []

for cycle in range(1, n_cycles + 1):
    print(f'\n=== Cycle {cycle} ===')
    
    # Year 1: make biparental crosses to create F1
    f1 = randCross(founders, nCrosses=n_crosses, nProgeny=n_f1_per_cross, simParam=SP)
    records.append(summarize_stage(cycle, 1, 'F1', f1))
    print(f'Year 1 F1 size: {len(f1)}')
    
    # Years 2-8: sequential selfing from F1 to F8
    f2 = self(f1, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 2, 'F2', f2))
    
    f3 = self(f2, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 3, 'F3', f3))
    
    f4 = self(f3, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 4, 'F4', f4))
    
    f5 = self(f4, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 5, 'F5', f5))
    
    f6 = self(f5, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 6, 'F6', f6))
    
    f7 = self(f6, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 7, 'F7', f7))
    
    f8 = self(f7, nProgeny=n_self_progeny, simParam=SP)
    records.append(summarize_stage(cycle, 8, 'F8', f8))
    print(f'Year 8 F8 size: {len(f8)}')
    
    # Year 9: preliminary trials
    prelim_trials = phenotype_stage(f8, varE=var_e, simParam=SP)
    records.append(summarize_stage(cycle, 9, 'prelim_trials', prelim_trials))
    advanced_trials = keep_top_fraction(prelim_trials, prelim_keep_frac, simParam=SP)
    records.append(summarize_stage(cycle, 9, 'advanced_trials_selected', advanced_trials))
    print(f'Year 9 selected to advanced: {len(advanced_trials)}')
    
    # Year 10: advanced trials
    advanced_trials = phenotype_stage(advanced_trials, varE=var_e, simParam=SP)
    records.append(summarize_stage(cycle, 10, 'advanced_trials_evaluated', advanced_trials))
    elite_trials_1 = keep_top_fraction(advanced_trials, advanced_keep_frac, simParam=SP)
    records.append(summarize_stage(cycle, 10, 'elite_trials_1_selected', elite_trials_1))
    print(f'Year 10 selected to elite 1: {len(elite_trials_1)}')
    
    # Year 11: elite trials 1
    elite_trials_1 = phenotype_stage(elite_trials_1, varE=var_e, simParam=SP)
    records.append(summarize_stage(cycle, 11, 'elite_trials_1_evaluated', elite_trials_1))
    elite_trials_2 = keep_top_fraction(elite_trials_1, elite1_keep_frac, simParam=SP)
    records.append(summarize_stage(cycle, 11, 'elite_trials_2_selected', elite_trials_2))
    print(f'Year 11 selected to elite 2: {len(elite_trials_2)}')
    
    # Year 12: elite trials 2
    elite_trials_2 = phenotype_stage(elite_trials_2, varE=var_e, simParam=SP)
    records.append(summarize_stage(cycle, 12, 'elite_trials_2_evaluated', elite_trials_2))
    
    # Year 13: release selection
    released_variety = selectInd(elite_trials_2, nInd=n_release, use='pheno', simParam=SP)
    records.append(summarize_stage(cycle, 13, 'released_variety', released_variety))
    released_lines.append(released_variety)
    print(f'Year 13 released lines: {len(released_variety)}')


## Convert Results to Tables

In [ ]:
results = pd.DataFrame(records)
results


## Stage-by-Stage Summary

The table below summarizes average population size, mean genetic value, and genetic variance across cycles for each stage.

In [ ]:
stage_summary = (
    results.groupby('stage', as_index=False)
    .agg({
        'n': 'mean',
        'meanG': 'mean',
        'varG': 'mean',
    })
)
stage_summary


## Plot Genetic Mean Across Stages

In [ ]:
plot_df = stage_summary.copy()
plot_df['stage_order'] = np.arange(len(plot_df))

plt.figure(figsize=(12, 5))
plt.plot(plot_df['stage_order'], plot_df['meanG'], marker='o')
plt.xticks(plot_df['stage_order'], plot_df['stage'], rotation=45, ha='right')
plt.ylabel('Mean Genetic Value')
plt.xlabel('Stage')
plt.title('Nannuru: Mean Genetic Value Across Pipeline Stages')
plt.tight_layout()
plt.show()


## Plot Genetic Variance Across Stages

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(plot_df['stage_order'], plot_df['varG'], marker='o', color='darkorange')
plt.xticks(plot_df['stage_order'], plot_df['stage'], rotation=45, ha='right')
plt.ylabel('Genetic Variance')
plt.xlabel('Stage')
plt.title('Nannuru: Genetic Variance Across Pipeline Stages')
plt.tight_layout()
plt.show()


## Released Variety Summary

This final table reports the released line from each simulated cycle.

In [ ]:
released_summary = results[results['stage'] == 'released_variety'].copy()
released_summary


## Interpretation

This notebook provides a direct AlphaSimPy interpretation of the BRAID abstraction for **Nannuru**.

Because the BRAID file omitted several numerical details, the notebook uses transparent assumptions for:
- founder population size,
- number of crosses,
- selfing progeny counts,
- executable genome dimensions.

The resulting notebook is intended as a **tutorial-style starting point** that can be refined once more program-specific design details become available.
